In [1]:
import time
start_time = time.time()

In [2]:
import sys
import numpy as np
from importlib.metadata import version, PackageNotFoundError

print("Python:", sys.executable)
print("Python version:", sys.version)
print("Loaded NumPy:", np.__version__)
print("NumPy path:", np.__file__)

for package in (
    "numpy", "scipy", "turbustat",
    "spectral-cube", "astropy", "setuptools",
):
    try:
        print(f"{package:15s}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package:15s}: NOT FOUND")

Python: C:\Users\javas\anaconda3\envs\viento\python.exe
Python version: 3.12.9 | packaged by conda-forge | (main, Mar  4 2025, 22:37:18) [MSC v.1943 64 bit (AMD64)]
Loaded NumPy: 2.3.0
NumPy path: C:\Users\javas\anaconda3\envs\viento\Lib\site-packages\numpy\__init__.py
numpy          : 1.26.4
scipy          : 1.14.1
turbustat      : 1.3
spectral-cube  : 0.6.7
astropy        : 7.1.0
setuptools     : 80.10.2


# Generate moment maps
Run all cells in order in the **viento** kernel. Configuration is read once per run.
Each `sig_E` generates one PPV cube, moment-0/1 maps, a JPG and a FITS file.
All diagnostic output is also written to `*_sanity_checks.txt` beside the input cubes.
Moment 2 remains on hold. Moment 0 is normalized only for plotting, never for FITS export.


In [3]:
from pathlib import Path
from copy import deepcopy
from contextlib import redirect_stdout
from io import StringIO
from datetime import datetime, timezone
from time import perf_counter
import importlib
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.constants import m_p, k_B
from astropy.io import fits
from turbustat.simulator import make_ppv
from spectral_cube import SpectralCube
import ppv_config
import synthetic_observations

importlib.reload(ppv_config)
importlib.reload(synthetic_observations)
from synthetic_observations import load_field_pair, print_pair_summary




C:\Users\javas\anaconda3\envs\viento\Lib\site-packages\turbustat\__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
C:\Users\javas\anaconda3\envs\viento\Lib\site-packages\dask\array\chunk_types.py:131: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.3.0)
  import scipy.sparse


ImportError: numpy.core.multiarray failed to import (auto-generated because you didn't call 'numpy.import_array()' after cimporting numpy; use '<void>numpy._import_array' to disable if you are certain you don't need it).

In [ ]:
line = 'nan'
pc = 1.0
pix = 1.0
bins = 0

## Load the pair and start the report
The report is overwritten when this cell is rerun, then updated after every diagnostic message.


In [ ]:
config = deepcopy(ppv_config.config)
physical_config = deepcopy(ppv_config.physical_config)
spectral_config = deepcopy(ppv_config.spectral_config)
# The former single-value sig_E setting is no longer used.
physical_config.pop("sig_E", None)
sig_E_values = [float(value) for value in physical_config["sig_E_values"]]
if not sig_E_values or not np.isfinite(sig_E_values).all():
    raise ValueError("sig_E_values must contain finite values.")
if len(set(sig_E_values)) != len(sig_E_values):
    raise ValueError("sig_E_values contains duplicates.")

In [ ]:
pair = load_field_pair(**config)
header_velocity = pair["header_velocity"]
header_density = pair["header_density"]
output_dir = Path(pair["paths"]["velocity"]).parent
velocity_name = Path(pair["paths"]["velocity"]).stem
sample_name = velocity_name.replace("_3d_velocity_", "_moments_")
summary_path = output_dir / f"{sample_name}_sanity_checks.txt"
summary_path.write_text("", encoding="utf-8")

def report(*items, sep=" ", end="\n"):
    """Show diagnostics and append the same text to the run report."""
    text = sep.join(str(item) for item in items) + end
    print(text, end="", flush=True)
    with summary_path.open("a", encoding="utf-8") as stream:
        stream.write(text)

report("Run started (UTC):", datetime.now(timezone.utc).isoformat())
report("Working directory:", Path.cwd())
report("Input configuration:", config)
report("Physical configuration:", physical_config)
report("Spectral configuration:", spectral_config)
report("Report:", summary_path.resolve())
with StringIO() as buffer:
    with redirect_stdout(buffer):
        print_pair_summary(pair)
    report(buffer.getvalue())


In [ ]:
# ---------------------------------------------------------
# Velocity: preserve the raw realization, then assign units
# ---------------------------------------------------------
velocity_scale = physical_config["velocity_scale_kms"]

velocity_64 = u.Quantity(
    np.ascontiguousarray(
        pair["velocity_base"] * velocity_scale,
        dtype=np.float64,
    ),
    unit=u.km / u.s,
)

# ---------------------------------------------------------
# Standardize the density-base field
# ---------------------------------------------------------
density_base = pair["density_base"]

density_norm = (
    (density_base - density_base.mean())
    / density_base.std(ddof=0)
)

# ---------------------------------------------------------

temperature = spectral_config["temperature_K"] * u.K
particle_mass = spectral_config["particle_mass_mp"] * m_p

sigma_thermal = np.sqrt(
    k_B * temperature / particle_mass
).to(u.km / u.s)


report("Velocity")
report(f"  Mean: {velocity_64.mean():.6f}")
report(f"  Std: {velocity_64.std():.6f}")
report(f"  Variance: {velocity_64.var():.6f}")
report(f"  All finite: {np.isfinite(velocity_64.value).all()}")
report("Standardized density-base field")
report(f"  Mean: {density_norm.mean():.6f}")
report(f"  Std: {density_norm.std():.6f}")
report(f"  All finite: {np.isfinite(density_norm).all()}")
report(f"Thermal dispersion: {sigma_thermal:.6f}")


In [ ]:
model_name = header_velocity["MODEL"].strip().lower()

length_scales = np.array([
    header_velocity["LSCALE_X"],
    header_velocity["LSCALE_Y"],
    header_velocity["LSCALE_Z"],
], dtype=float)

if not np.allclose(length_scales, length_scales[0]):
    raise ValueError("A single R0 assumes isotropic length scales.")

L = float(length_scales[0])

# Assumes the default GSTools rescale used during field generation.
if model_name == "exponential":
    alpha = 1.0
    rescale = 1.0
elif model_name == "gaussian":
    alpha = 2.0
    rescale = np.sqrt(np.pi) / 2.0
elif model_name == "stable":
    alpha = float(header_velocity["M3D"])
    rescale = 1.0
else:
    raise ValueError(f"Unsupported model: {model_name}")

r0_input = float((L / rescale) * np.log(2.0)**(1.0 / alpha))

report("\nInput covariance scale")
report(f"  Model: {model_name}")
report(f"  Length scale: {L:.6f}")
report(f"  Rescale: {rescale:.6f}")
report(f"  R0: {r0_input:.6f}")
report("  Definition: input correlation equals 0.5")
report("  Units: same as input grid spacing")

## Diagnostic comparison
Compare moment-0 morphology after dividing both maps by their means over the same valid pixels.
This comparison does not check the absolute intensity conversion. Differences are PPV minus direct LOS values;
no automatic pass/fail tolerance is imposed on these differences.


In [ ]:
def check_moments(sb, vv, emissivity_64):
    # ---------------------------------------------------------
    # Direct LOS moments from the input fields
    # ---------------------------------------------------------
    los_axis = pair["los_axis"]

    weights = emissivity_64.value
    vel = velocity_64.to_value(u.km / u.s)

    # Integrated emission weights.
    sb_direct = np.sum(weights, axis=los_axis)

    # Weighted centroid.
    vv_direct = (
        np.sum(weights * vel, axis=los_axis)
        / sb_direct
    )

    # ---------------------------------------------------------
    # Compare PPV moments against direct LOS calculations
    # ---------------------------------------------------------
    if not (sb.shape == vv.shape == sb_direct.shape):
        raise ValueError("PPV and direct moment maps have different shapes.")

    valid = (
        np.isfinite(sb)
        & np.isfinite(vv)
        & np.isfinite(sb_direct)
        & np.isfinite(vv_direct)
        & (sb > 0)
        & (sb_direct > 0)
    )

    if not valid.any():
        raise ValueError("No valid pixels available for comparison.")

    # Compare moment-0 morphology, removing its overall conversion factor.
    # Both maps have mean = 1 over the same valid pixels.
    sb_norm = sb[valid] / sb[valid].mean()
    sb_direct_norm = sb_direct[valid] / sb_direct[valid].mean()

    differences = {
        "Moment 0 (normalized)": sb_norm - sb_direct_norm,
        "Moment 1 (km/s)": vv[valid] - vv_direct[valid],
    }

    report(f"Compared pixels: {valid.sum()} / {valid.size}\n")
    report(f"{'Quantity':26s} {'Mean difference':>16s} {'RMS':>13s} {'Max absolute':>14s}")

    for label, difference in differences.items():
        report(
            f"{label:26s} "
            f"{difference.mean():16.6e} "
            f"{np.sqrt(np.mean(difference**2)):13.6e} "
            f"{np.max(np.abs(difference)):14.6e}"
        )


## Generate and inspect every emissivity contrast


In [ ]:
results = {}

for sig_E in sig_E_values:
    report(f"\nsig_E = {sig_E:g}")

    # Relative emissivity: mean = 1 for every experiment.
    emissivity_relative = np.exp(sig_E * density_norm)
    emissivity_relative /= emissivity_relative.mean()

    emissivity_64 = u.Quantity(
        np.ascontiguousarray(emissivity_relative, dtype=np.float64),
        unit=u.cm**-3,
    )

    report(f"  Emissivity mean: {emissivity_relative.mean():.6f}")
    report(f"  Std(log emissivity): {np.log(emissivity_relative).std():.6f}")
    report(f"  All positive: {np.all(emissivity_relative > 0)}")
    report(f"  All finite: {np.isfinite(emissivity_relative).all()}")
    report(f"  Emissivity input mean: {emissivity_64.mean():.6f}")
    report(f"  Emissivity input unit: {emissivity_64.unit}")
    if not (np.isfinite(emissivity_relative).all() and np.all(emissivity_relative > 0)):
        raise ValueError("Emissivity must be finite and positive; see report.")

    start = perf_counter()

    cube_hdu = make_ppv(
        velocity_64,
        emissivity_64,
        los_axis=pair["los_axis"],
        m=particle_mass,
        T=temperature,
        los_length=spectral_config["los_length_pc"] * u.pc,
        chan_width=spectral_config["channel_width_kms"] * u.km / u.s,
        v_min=spectral_config["v_min_kms"] * u.km / u.s,
        v_max=spectral_config["v_max_kms"] * u.km / u.s,
        threads=1,
    )

    # Save the full PPV cube, including its spectral and spatial headers.
    ppv_name = velocity_name.replace("_3d_velocity_", "_ppv_")
    ppv_path = output_dir / f"{ppv_name}_sigE{sig_E:g}.fits"

    cube_hdu.header["SIGE"] = (float(sig_E), "Emissivity contrast parameter")
    cube_hdu.header["MODEL"] = header_velocity["MODEL"]
    cube_hdu.header["M3D"] = header_velocity["M3D"]
    cube_hdu.header["LOSAXIS"] = pair["los_axis"]
    cube_hdu.header["SEED1"] = header_density["SEED"]
    cube_hdu.header["SEED2"] = header_velocity["SEED"]

    cube_hdu.writeto(ppv_path, overwrite=True)
    report(f"  Saved PPV cube: {ppv_path.name}")
    
    cube = SpectralCube.read(cube_hdu).with_spectral_unit(u.km / u.s)

    moment0 = cube.moment0()
    moment1 = cube.moment1()

    sb = moment0.value
    vv = moment1.value

    report(f"  PPV shape: {cube.shape} (spectral, spatial, spatial)")
    report(f"  Intensity unit: {cube.unit}")
    report(f"  First channel center: {cube.spectral_axis[0]}")
    report(f"  Last channel center: {cube.spectral_axis[-1]}")
    if len(cube.spectral_axis) > 1:
        report(f"  Channel spacing: {cube.spectral_axis[1] - cube.spectral_axis[0]}")
    report(f"  Moment 0 shape: {sb.shape}")
    report(f"  Moment 1 shape: {vv.shape}")
    report(f"  Moment 0 finite: {np.isfinite(sb).all()}")
    report(f"  Moment 0 positive: {np.all(sb > 0)}")
    report(f"  Moment 1 finite: {np.isfinite(vv).all()}")
    if not np.isfinite(vv).any():
        raise ValueError("The centroid map contains no finite values.")
    check_moments(sb, vv, emissivity_64)

    # Retain the 2D results for comparison and later export.
    results[sig_E] = {
        "sig_E": sig_E,
        "moment0": moment0,
        "moment1": moment1,
        "sb": sb,
        "vv": vv,
        "sig": float(np.nanstd(vv)),
        "sig2": float(np.nanvar(vv)),
    }

    report(f"  Completed in {perf_counter() - start:.2f} s")
    report(f"  Mean relative emissivity: {emissivity_relative.mean():.6f}")
    report(f"  Mean moment 0:            {np.nanmean(sb):.6f}")
    report(f"  Centroid-map std:         {results[sig_E]['sig']:.6f}")
    report(f"  Centroid-map variance:    {results[sig_E]['sig2']:.6f}")

    fig, axes = plt.subplots(
        1, 2,
        figsize=(11, 4.5),
        constrained_layout=True,
    )

    for ax, data, title, cmap, limits in zip(
        axes,
        (sb / sb.max(), vv),
        ("Moment 0 (normalized)", "Moment 1"),
        ("magma", "RdBu_r"),
        ({"vmin": 0, "vmax": 1}, {}),
    ):
        im = ax.imshow(
            data,
            origin="lower",
            cmap=cmap,
            **limits,
        )

        fig.colorbar(im, ax=ax)
        ax.set_title(title)
        ax.set_xlabel("Pixel")
        ax.set_ylabel("Pixel")

    fig.suptitle(f"sig_E = {sig_E:g}")

    figure_path = output_dir / f"{velocity_name}_sigE{sig_E:g}_moments.jpg"

    fig.savefig(figure_path, dpi=300, facecolor="white")
    plt.show()
    plt.close(fig)

    report(f"  Saved: {figure_path.name}")


## Export FITS maps


In [ ]:
saved_paths = []

for sig_E, entry in results.items():

    filename = f"{sample_name}_sigE{sig_E:g}.fits"
    output_path = output_dir / filename

    # -------------------------------------------------------
    # Primary header: experiment metadata
    # -------------------------------------------------------
    primary = fits.PrimaryHDU()
    header = primary.header

    ny, nx = entry["vv"].shape


    pixel_scale = pix * (2**bins)
    box_size = np.hypot(nx, ny) * pc * pixel_scale* (2**bins)

    header["PC"] = (pc, "Synthetic scale conversion; unity")
    header["PIX"] = (pixel_scale, "Pixel scale in synthetic grid units")
    header["BINSIZE"] = (bins, "Spatial binning exponent; factor = 2**BINSIZE")
    header["BOX_SIZE"] = (float(box_size), "Full map diagonal in synthetic grid units")

    header["S0"] = (0.0, "No applied seeing")
    header["NOISE"] = (0.0, "No added instrumental noise")

    header["FILE"] = (filename, "FITS file name")
    header["MODEL"] = (header_velocity["MODEL"], "Input covariance model")
    header["M3D"] = (header_velocity["M3D"], "Input model shape parameter")
    header["SIGE"] = (float(sig_E), "Log-emissivity contrast parameter")

    header["SEED1"] = (header_density["SEED"], "Density-base seed")
    header["SEED2"] = (header_velocity["SEED"], "Velocity-base seed")

    header["VELFILE"] = (
        Path(pair["paths"]["velocity"]).name,
        "Input velocity cube",
    )
    header["DENFILE"] = (
        Path(pair["paths"]["density"]).name,
        "Input density-base cube",
    )

    # Input grid and covariance geometry
    for axis, size in zip("XYZ", pair["shape"]):
        header[f"N{axis}"] = (int(size), f"Input grid size along {axis}")

    for key in (
        "LSCALE_X", "LSCALE_Y", "LSCALE_Z",
        "ANGLE_X", "ANGLE_Y", "ANGLE_Z",
        "DX", "DY", "DZ",
    ):
        header[key] = (
            header_velocity[key],
            header_velocity.comments[key],
        )

    header["LOSAXIS"] = (pair["los_axis"], "Input NumPy LOS axis: 0=X, 1=Y, 2=Z")

    # Physical and spectral settings used for this batch
    header["VSCALE"] = (
        physical_config["velocity_scale_kms"],
        "Velocity multiplier in km/s",
    )
    header["TEMP"] = (spectral_config["temperature_K"], "Gas temperature in K")
    header["MASSMP"] = (
        spectral_config["particle_mass_mp"],
        "Particle mass in proton masses",
    )
    header["LOS_PC"] = (
        spectral_config["los_length_pc"],
        "Line-of-sight depth in pc",
    )
    header["CHANWID"] = (
        spectral_config["channel_width_kms"],
        "Requested channel width in km/s",
    )
    header["VMIN"] = (spectral_config["v_min_kms"], "Requested lower edge in km/s")
    header["VMAX"] = (spectral_config["v_max_kms"], "Requested upper edge in km/s")

    # Statistics specific to this sig_E
    header["SIG"] = (entry["sig"], "Centroid-map std in km/s")
    header["SIG2"] = (entry["sig2"], "Centroid-map variance in (km/s)^2")

    header["R0"] = (
        r0_input,
        "Input 3D half-correlation scale in grid units",
    )
    header["R0DEF"] = ("CORR=0.5", "Definition of input R0")
    header["RESCALE"] = (rescale, "GSTools covariance rescale factor")
    
    header.add_history("Density-base field standardized before exponentiation.")
    header.add_history("Emissivity = exp(sig_E * density_norm), normalized to mean 1.")
    header.add_history("MOM0 divided by its maximum before export; peak = 1.")

    # -------------------------------------------------------
    # Image extensions: preserve moment headers and units
    # -------------------------------------------------------
    sb_export = entry["sb"] / np.nanmax(entry["sb"])

    mom0_header = entry["moment0"].header.copy()
    mom0_header["BUNIT"] = ("1", "Dimensionless normalized intensity")
    mom0_header["NORM"] = ("MAX", "Divided by the map maximum")
    mom0_header["NORMFAC"] = (
        float(np.nanmax(entry["sb"])),
        "Original map maximum before normalization",
    )

    mom0_hdu = fits.ImageHDU(
        data=sb_export,
        header=mom0_header,
        name="MOM0",
    )

    mom1_hdu = fits.ImageHDU(
        data=entry["vv"],
        header=entry["moment1"].header.copy(),
        name="MOM1",
    )

    fits.HDUList(
        [primary, mom0_hdu, mom1_hdu]
    ).writeto(output_path, overwrite=True)

    saved_paths.append(output_path)
    report(f"Saved: {output_path.name}")


## Verify every exported file
Check the saved arrays against the in-memory results and include the FITS extension inventory in the report.


In [ ]:
for output_path, (sig_E, entry) in zip(saved_paths, results.items()):
    with fits.open(output_path) as hdul:
        report(f"\nExport verification: {output_path.name}")
        with StringIO() as buffer:
            hdul.info(output=buffer)
            report(buffer.getvalue())
        report("  sig_E:", hdul[0].header["SIGE"])
        report("  SIG2:", hdul[0].header["SIG2"])
        for extension, key in (("MOM0", "sb"), ("MOM1", "vv")):
            expected = entry[key]
            if extension == "MOM0":
                expected = expected / np.nanmax(expected)

            matches = np.array_equal(
                hdul[extension].data, expected, equal_nan=True
            )
            report(f"  {extension}: shape={hdul[extension].data.shape}, "
                   f"unit={hdul[extension].header.get('BUNIT', '')}, "
                   f"matches in-memory map={matches}")
            if not matches:
                raise ValueError(f"Export mismatch: {output_path.name}, {extension}")
report("\nRun complete. Exported FITS files:", len(saved_paths))
report("Sanity-check report saved:", summary_path.resolve())


In [ ]:
print("--- %s seconds ---" % (time.time() - start_time))